# Chavruta.AI — LoRA משגר (Colab / Kaggle / Lightning)

מחברת **משגר דקה** בלבד — כל לוגיקת האימון ב-`scripts/train_lora.py` (מקור אמת יחיד).

⚠️ **חובה:** `Runtime → Change runtime type → **GPU (T4)**` — לא TPU ולא CPU.
ה-stack (Unsloth + bitsandbytes) הוא CUDA-only ולא ירוץ על TPU.

💾 **Checkpoints נשמרים מקומית** ב-`/content/outputs/` (לא Drive). זכור: אחסון מקומי
ב-Colab הוא זמני — התא האחרון מוריד את ה-adapter למחשב שלך בסוף.

### 1. ודא שיש GPU

In [ ]:
!nvidia-smi

### 2. התקנה (פעם אחת לסשן)

In [ ]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate bitsandbytes datasets

### 3. הבא את הקוד + הדאטה

מלא את `REPO_URL` אם דחפת ל-GitHub. אחרת — העלה ידנית את
`scripts/train_lora.py` ושני קבצי ה-`.jsonl` (דרך הסרגל משמאל → Upload),
ודלג על ה-clone.

In [ ]:
import os

REPO_URL = ""  # למשל: https://github.com/<user>/Chavruta.AI.git  (השאר ריק אם מעלים ידנית)
PROJECT = "/content/Chavruta.AI"

if REPO_URL:
    if not os.path.exists(PROJECT):
        !git clone $REPO_URL $PROJECT
    %cd $PROJECT
else:
    # מצב העלאה ידנית: ודא שהמבנה קיים
    os.makedirs("/content/scripts", exist_ok=True)
    os.makedirs("/content/data/processed", exist_ok=True)
    %cd /content
    print("העלה ידנית: scripts/train_lora.py , data/processed/torah_mixed_train.jsonl , data/processed/torah_mixed_val.jsonl")

# בדיקה שהקבצים קיימים
for p in ["scripts/train_lora.py", "data/processed/torah_mixed_train.jsonl", "data/processed/torah_mixed_val.jsonl"]:
    print(("✅" if os.path.exists(p) else "❌"), p)

### 4. אימון — checkpoint מקומי ב-`/content/outputs`

שנה פרמטרים דרך ה-flags. אם OOM: הוסף `--max_seq 1024` או `--bs 1 --grad_accum 8`.

In [ ]:
!python scripts/train_lora.py \
  --model Qwen/Qwen3.5-4B \
  --epochs 1 \
  --max_seq 2048 \
  --bs 2 --grad_accum 4 \
  --out /content/outputs/chavruta-qwen35-4b-lora

### 5. הורד את ה-adapter (אחסון מקומי זמני — הורד לפני ניתוק!)

ה-adapter קטן (כמה MB). התא מכווץ אותו ומוריד למחשב שלך.

In [ ]:
import shutil
adapter_dir = "/content/outputs/chavruta-qwen35-4b-lora"
zip_path = shutil.make_archive("/content/chavruta_adapter", "zip", adapter_dir)
print("📦", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("לא Colab או אין דפדפן — הורד ידנית מתוך הסרגל:", zip_path)

---
### הערות
- **למה לא Drive:** לפי בקשתך — ה-checkpoints מקומיים ב-`/content`. המחיר: זה זמני
  (ניתוק/סגירת Colab = אובדן). לכן תא 5 מוריד את ה-adapter. לריצות ארוכות
  שקול להוסיף `--out /content/drive/MyDrive/...` אחרי mount.
- **Qwen3.5 חדש:** אם Unsloth לא מזהה — עדכן לאחרון. אם תבנית ה-chat שונה
  מ-ChatML — תקן את המסמנים ב-`train_on_responses_only` בתוך `train_lora.py`.
- **Kaggle/Lightning:** אותה מחברת עובדת; רק תא 5 (files.download) הוא Colab-ספציפי —
  ב-Kaggle שמור ל-`/kaggle/working/` והורד מלשונית ה-Output.